# Hybrid Retrieval Eval (BM25 + Dense via RRF)

Fuses the lexical (BM25) and dense (Titan+FAISS) retrievers with Reciprocal
Rank Fusion. RRF combines by *rank position* (not raw score), so BM25's and
cosine's incomparable score scales don't matter. Hybrid typically beats both
arms because it gets lexical precision AND semantic recall.

Scored with the same metrics as bm25_eval / dense_eval, so it drops straight
into the comparison table.

**Prereqs:** `rag_eval/` uploaded (now includes `hybrid.py`); chunks + FAISS
index in S3 (from data_processing.ipynb dense ingestion).

## 1. Session + deps

In [ ]:
import sys
!{sys.executable} -m pip install -q rank-bm25==0.2.2 faiss-cpu==1.15.1
sys.path.insert(0, ".")
import io, os, tempfile
import boto3, numpy as np, pandas as pd, faiss

REGION = "us-east-1"; BUCKET = "rag-arch-eval-data-875699230934"
s3 = boto3.client("s3", region_name=REGION)
def read_parquet_s3(key):
    return pd.read_parquet(io.BytesIO(s3.get_object(Bucket=BUCKET, Key=key)["Body"].read()))

## 2. Load chunks, FAISS index, queries

Same inputs as the other eval notebooks. (Switch QKEY to the degraded set to
run hybrid on degraded queries too.)

In [ ]:
from rag_eval.chunking import Chunk
chunks_df = read_parquet_s3("chunks/chunks.parquet")
chunks = [Chunk(r.chunk_id, r.doc_id, r.text) for r in chunks_df.itertuples()]

with tempfile.TemporaryDirectory() as d:
    local = os.path.join(d, "faiss.index")
    s3.download_file(BUCKET, "indexes/faiss.index", local)
    index = faiss.read_index(local)
id_map = read_parquet_s3("indexes/id_map.parquet").sort_values("row")
row_to_doc = id_map["doc_id"].tolist()

# Toggle which query set to evaluate. The two parquets have DIFFERENT schemas:
#   original: column 'query'          + 'evidence_list' (list of {title})
#   degraded: column 'degraded_query'  + 'evidence_titles' (list of titles)
USE_DEGRADED = True   # <-- set False for the original queries

qrecs = []
if USE_DEGRADED:
    dq = read_parquet_s3("data/degraded_queries.parquet")
    for r in dq.itertuples():
        if r.degraded_query:  # skip any that failed to generate
            qrecs.append({"query": r.degraded_query, "question_type": r.question_type,
                          "rel": set(r.evidence_titles)})
else:
    q_df = read_parquet_s3("data/queries.parquet")
    for r in q_df.itertuples():
        rel = tuple(dict.fromkeys(e["title"] for e in r.evidence_list))
        if rel:
            qrecs.append({"query": r.query, "question_type": r.question_type, "rel": set(rel)})
print(f"USE_DEGRADED={USE_DEGRADED} | chunks: {len(chunks)} | index: {index.ntotal} | queries: {len(qrecs)}")

## 3. Build both arms + a wrapped dense that uses the prebuilt FAISS index

`HybridRetriever` composes a BM25 arm and a dense arm. BM25 we index from the
chunks directly. For dense we reuse the FAISS index already in S3 (no
re-embedding) by wrapping it in a tiny Retriever-shaped object the hybrid can
call.

In [ ]:
from rag_eval.retrievers import BM25Retriever, HybridRetriever, ranked_doc_ids
from rag_eval.retrievers.base import Hit
from rag_eval.bedrock import TitanEmbedder

bm25 = BM25Retriever(); bm25.index(chunks)

def _norm(m):
    n = np.linalg.norm(m, axis=1, keepdims=True); n[n==0]=1.0; return m/n

class PrebuiltDense:
    """Minimal dense arm backed by the already-built FAISS index.
    Reuses ONE cached TitanEmbedder; chunks are in the same order as id_map rows."""
    name = "dense_bedrock"
    def __init__(self):
        self.emb = TitanEmbedder(model_id="amazon.titan-embed-text-v2:0", region=REGION)
        self._chunk_by_pos = chunks
    def index(self, chunks):
        pass  # index already built in ingestion
    def retrieve(self, query, k):
        v = _norm(self.emb.embed_query(query).reshape(1, -1).astype(np.float32))
        _, idx = index.search(v, min(k, index.ntotal))
        return [Hit(self._chunk_by_pos[i], 0.0) for i in idx[0] if i != -1]

hybrid = HybridRetriever(rrf_k=60, bm25=bm25, dense=PrebuiltDense())
print(ranked_doc_ids(hybrid.retrieve(qrecs[0]["query"], 10))[:5])

## 4. Score hybrid

In [ ]:
from rag_eval.metrics import METRICS
KS = [1,3,5,10,20]; FETCH = max(KS)*5

rows = []
for j, q in enumerate(qrecs, 1):
    ranked = ranked_doc_ids(hybrid.retrieve(q["query"], FETCH))
    row = {"question_type": q["question_type"]}
    for k in KS:
        for name, fn in METRICS.items():
            row[f"{name}@{k}"] = fn(ranked, q["rel"], k)
    rows.append(row)
    if j % 500 == 0: print(f"  {j}/{len(qrecs)}")
per_q = pd.DataFrame(rows)
print("scored", len(per_q), "queries")

## 5. Results — compare to BM25 and dense

Put this row next to the BM25 and dense rows. Expectation: hybrid ≥ best of
the two individual arms on recall, and more robust across question types.

In [ ]:
cols = [c for c in per_q.columns if "@" in c]
print("=== HYBRID (RRF) OVERALL ===")
print(per_q[cols].mean().round(4).to_string())
print("\n=== HYBRID by question type (recall@10 / all_hit@10) ===")
bt = per_q.groupby("question_type")[cols].mean().round(4)
print(bt[["recall@10","all_hit@10","mrr@10","ndcg@10"]].to_string())

## 6. Save

In [ ]:
RUN = "hybrid_rrf_degraded" if USE_DEGRADED else "hybrid_rrf_notebook"
summary = {"run": RUN, "n_queries": len(per_q)}
summary.update(per_q[cols].mean().round(4).to_dict())
for df, key in [(per_q, f"results/notebook/{RUN}_per_query.csv"),
                (pd.DataFrame([summary]), f"results/notebook/{RUN}_summary.csv")]:
    b = io.StringIO(); df.to_csv(b, index=False)
    s3.put_object(Bucket=BUCKET, Key=key, Body=b.getvalue().encode())
    print("wrote s3://%s/%s" % (BUCKET, key))